# 06 · Serve it with FastAPI
- `app/main.py` loads the logs, the index and the model once at start-up
- Endpoints: `/health`, `/ready`, `/stats`, `POST /v1/chat`, `POST /v2/ask` (answer + sources), `POST /v2/ingest`
- Here we start the server in the background and call it like any client would

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'          # models come from the local cache only
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_colwidth', 120)
import subprocess, sys, time, httpx
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app', '--port', '8765'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})
api = httpx.Client(base_url='http://127.0.0.1:8765', timeout=120)
for _ in range(240):
    try:
        if api.get('/ready').status_code == 200: break
    except httpx.TransportError: pass
    time.sleep(1)
api.get('/ready').json()

{'status': 'ready',
 'model': 'Qwen/Qwen2.5-0.5B-Instruct',
 'device': 'mps',
 'chunks': 100,
 'startup_seconds': 1.2}

In [2]:
api.get('/health').json(), api.get('/stats').json()

({'status': 'ok'},
 {'from': '2026-10-13 00:00:01+00:00',
  'to': '2026-10-15 23:59:37+00:00',
  'requests': 28647,
  'http_5xx': 84,
  'http_4xx': 866,
  'app_errors': 78,
  'app_warnings': 23,
  'top_exceptions': {'java.sql.SQLTransientConnectionException': 42,
   'java.lang.NullPointerException': 27,
   'AH00128': 12,
   'AH01102': 9,
   'AH00957': 9}})

## v1: chat with a session

In [3]:
for m in ['What is happening in the latest logs?', 'What should I do about it?']:
    r = api.post('/v1/chat', json={'session_id': 'nag', 'message': m}).json()
    print('YOU:', m); print('BOT:', r['answer'], f"({r['latency_ms']} ms)"); print()

YOU: What is happening in the latest logs?
BOT: The logs indicate that there have been 28647 requests made to the API endpoint `/api/login`. This suggests that the application has successfully processed at least 28,647 login requests. The logs also show that the request times range from 77ms to 112ms, indicating that some requests were successful while others were unsuccessful or took longer than expected. There are no apparent errors or exceptions recorded in these logs, which means that the system appears to be functioning correctly. (1309 ms)



YOU: What should I do about it?
BOT: Since there are no errors or exceptions recorded in the logs, it seems like the system is functioning as intended. However, if you need assistance with any specific issue or want to report a bug, please feel free to let me know. (605 ms)



## v2: answer with sources

In [4]:
r = api.post('/v2/ask', json={'question': 'What error started after the v1.4.2 deploy and how was it fixed?'}).json()
print(r['answer']); pd.DataFrame(r['sources'])[['id', 'file', 'start', 'score', 'text']]

The error that started after the v1.4.2 deploy was due to a `NullPointerException`. The fix was to roll back to the previous version (`deploy rollback shop-app`) before deploying the changes.


,id,file,start,score,text
0,S1,kb/runbooks/bad-deploy-rollback.md,,0.591,# Runbook: errors right after a deploy
1,S2,deploys.log,2026-10-15 14:40,0.546,deploys.log 2026-10-15 14:40–2026-10-15 14:50:
2,S3,deploys.log,2026-10-13 09:00,0.518,deploys.log 2026-10-13 09:00–2026-10-13 09:10:
3,S4,app/application.log,2026-10-13 09:00,0.517,app/application.log 2026-10-13 09:00–2026-10-13 09:10:
4,S5,deploys.log,2026-10-15 14:00,0.432,deploys.log 2026-10-15 14:00–2026-10-15 14:10:
5,S6,app/application.log,2026-10-15 14:00,0.431,app/application.log 2026-10-15 14:00–2026-10-15 14:10:


## Filters: one file, one time range

In [5]:
r = api.post('/v2/ask', json={'question': 'What went wrong?', 'file': 'tomcat', 'since': '2026-10-14 08:00', 'until': '2026-10-14 12:00'}).json()
print(r['answer']); [s['file'] + ' ' + s['start'] for s in r['sources']]

The issue occurred due to insufficient memory allocation by the Java Virtual Machine (JVM), specifically leading to a `java.lang.OutOfMemoryError: Java heap space`. This error was reported multiple times across different log files, indicating that the JVM had exhausted its available heap space. The specific instances where this error occurred were logged as:

- First occurrence: 2026-10-14 11:02:10, last 2026-10-14 11:02:10, 1 line in tomcat/catalina.out
- Second occurrence: 2026-10-14 11:02:15, last 2026-10-1


['tomcat/catalina.out 2026-10-14 11:00',
 'app/application.log 2026-10-14 11:00',
 'apache/error_log 2026-10-14 11:00']

## Ingest a new log file, then ask about it

In [6]:
new = '\n'.join([
    '2026-10-16 09:12:01.100 ERROR 4512 --- [nio-8080-exec-3] c.shop.files.UploadService : java.io.IOException: No space left on device',
    '2026-10-16 09:12:04.300 ERROR 4512 --- [nio-8080-exec-9] c.shop.files.UploadService : java.io.IOException: No space left on device',
    '2026-10-16 09:13:00.000  WARN 4512 --- [scheduling-1] c.shop.ops.DiskMonitor : /var/lib/shop disk 100% full'])
print(api.post('/v2/ingest', json={'name': 'application-2026-10-16.log', 'text': new}).json())
r = api.post('/v2/ask', json={'question': 'Was there a disk full error?'}).json()
print(r['answer']); [s['file'] for s in r['sources'][:3]]

{'added_chunks': 1, 'total_chunks': 101}


Yes, there was a disk full error. The log entry shows that the disk was at 100% capacity, which led to the OutOfMemoryError.


['ingested/application-2026-10-16.log',
 'tomcat/catalina.out',
 'app/application.log']

## Bad input is rejected

In [7]:
api.post('/v2/ask', json={'question': ''}).status_code

422

In [8]:
assert api.get('/health').json() == {'status': 'ok'}
assert any(s['file'].startswith('ingested/') for s in r['sources'])
server.terminate(); server.wait()
print('✓ FastAPI serves v1 and v2; new logs are searchable right after ingest')

✓ FastAPI serves v1 and v2; new logs are searchable right after ingest
